# A Recurrent Network for Sentiment Classification

In [ ]:
import re
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import nltk
import torch
import torch.nn as nn

## Many-to-One: One Label for a Whole Text

$$
P(\text{positive} \mid w_1, \ldots, w_T) = \sigma\left( h_T W_o + b_o \right),
$$

### Data and Preprocessing

In [ ]:
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split

nltk.download("stopwords", quiet=True)

reviews = pd.read_csv("https://github.com/febse/data/raw/refs/heads/main/ta/reviews.csv")
reviews = reviews.dropna(subset=["Reviews", "Rating"])
reviews = reviews[reviews["Rating"] != 3]

review_texts = reviews["Reviews"].astype(str).tolist()
sentiment = (reviews["Rating"] > 3).astype(int).to_numpy()

print("reviews:", len(sentiment), " share of positive reviews:", sentiment.mean().round(3))
print(reviews["Rating"].value_counts().sort_index().to_dict())

In [ ]:
NEGATIONS = {"no", "nor", "not"}          # the negations in the stop list of nltk
STOP_WORDS = set(stopwords.words("english"))
stemmer = PorterStemmer()
stem_cache = {}

def stem(word):
    if word not in stem_cache:
        stem_cache[word] = stemmer.stem(word)
    return stem_cache[word]

def preprocess(text, keep_negations=True):
    text = text.lower().replace("won't", "will not").replace("can't", "can not").replace("n't", " not")
    words = re.findall(r"[a-z]+", text)
    kept = [w for w in words if w not in STOP_WORDS or (keep_negations and w in NEGATIONS)]
    return [stem(w) for w in kept]

print(review_texts[1][:150])
print(preprocess(review_texts[1])[:25])
print(preprocess("The battery isn't good and I can't recommend it, no value for money"))

In [ ]:
MAX_WORDS = 100
MAX_VOCAB = 30000

def build_dataset(keep_negations=True, seed=0):
    docs = [preprocess(text, keep_negations) for text in review_texts]
    idx_tr, idx_rest = train_test_split(np.arange(len(docs)), test_size=0.3, random_state=seed, stratify=sentiment)
    idx_va, idx_te = train_test_split(idx_rest, test_size=0.5, random_state=seed, stratify=sentiment[idx_rest])

    stem_counts = Counter(w for i in idx_tr for w in docs[i])
    stems = ["<pad>", "<unk>"] + [w for w, _ in stem_counts.most_common(MAX_VOCAB)]
    stem_to_id = {w: i for i, w in enumerate(stems)}

    matrix = np.zeros((len(docs), MAX_WORDS), dtype=np.int64)
    lengths = np.zeros(len(docs), dtype=np.int64)
    for i, doc in enumerate(docs):
        encoded = [stem_to_id.get(w, 1) for w in doc][:MAX_WORDS] or [1]    # an empty review becomes one <unk>
        matrix[i, :len(encoded)] = encoded
        lengths[i] = len(encoded)
    return {"docs": docs, "stems": stems, "stem_to_id": stem_to_id, "X": torch.tensor(matrix),
            "lengths": torch.tensor(lengths), "y": torch.tensor(sentiment, dtype=torch.float32),
            "train": idx_tr, "val": idx_va, "test": idx_te, "n_distinct": len(stem_counts)}

review_data = build_dataset()
doc_lengths = np.array([len(d) for d in review_data["docs"]])

print("distinct stems in the training reviews:", review_data["n_distinct"], "-> vocabulary size (with <pad>, <unk>):", len(review_data["stems"]))
print("most common stems:", review_data["stems"][2:17])
print(f"median length: {np.median(doc_lengths):.0f} words, reviews cut at {MAX_WORDS} words: {(doc_lengths > MAX_WORDS).mean():.1%}")
print("train / val / test:", len(review_data["train"]), len(review_data["val"]), len(review_data["test"]))

### The Network

### Gradient Clipping

$$
g \leftarrow \begin{cases} g & \text{if } \lVert g \rVert_2 \le c \\[4pt] c \, \dfrac{g}{\lVert g \rVert_2} & \text{if } \lVert g \rVert_2 > c \end{cases}
$$

In [ ]:
class SentimentNet(nn.Module):
    def __init__(self, vocab_size, kind="rnn", embedding_size=64, hidden_size=64):
        super().__init__()
        self.kind = kind
        self.embedding = nn.Embedding(vocab_size, embedding_size, padding_idx=0)
        layer = {"rnn": nn.RNN, "gru": nn.GRU, "lstm": nn.LSTM}[kind]
        self.recurrent = layer(embedding_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, word_ids, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(self.embedding(word_ids), lengths, batch_first=True, enforce_sorted=False)
        _, last_state = self.recurrent(packed)
        if self.kind == "lstm":
            last_state = last_state[0]                      # the LSTM also returns its cell state
        return self.output(last_state[-1]).squeeze(1)       # one logit per review

    def logits_after_each_word(self, word_ids):
        # for ONE review (1, T): the logit that we would get if the review ended after word t
        states, _ = self.recurrent(self.embedding(word_ids))
        return self.output(states[0]).squeeze(1)

In [ ]:
bce_loss = nn.BCEWithLogitsLoss()

def split_tensors(data, name):
    rows = torch.tensor(data[name])
    return data["X"][rows], data["lengths"][rows], data["y"][rows]

def predict_logits(model, X, lengths, batch_size=512):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(X[s:s + batch_size], lengths[s:s + batch_size]) for s in range(0, len(X), batch_size)])

def fit_sentiment(model, data, n_epochs=8, learning_rate=2e-3, batch_size=64, clip=5.0, seed=0):
    X_tr, len_tr, y_tr = split_tensors(data, "train")
    X_va, len_va, y_va = split_tensors(data, "val")
    torch.manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = {"val": []}
    best_loss, best_state = float("inf"), None
    for epoch in range(n_epochs):
        model.train()
        order = torch.randperm(len(X_tr))
        for start in range(0, len(order), batch_size):
            batch = order[start:start + batch_size]
            optimizer.zero_grad()
            bce_loss(model(X_tr[batch], len_tr[batch]), y_tr[batch]).backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()
        history["val"].append(bce_loss(predict_logits(model, X_va, len_va), y_va).item())
        if history["val"][-1] < best_loss:
            best_loss, best_state, history["best_epoch"] = history["val"][-1], {k: v.clone() for k, v in model.state_dict().items()}, epoch
    model.load_state_dict(best_state)
    return model, history

def test_scores(model, data):
    X_te, len_te, y_te = split_tensors(data, "test")
    probs = torch.sigmoid(predict_logits(model, X_te, len_te)).numpy()
    return probs, y_te.numpy()

### Training and Comparison

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

sentiment_models = {}
for kind in ["rnn", "gru"]:
    torch.manual_seed(0)
    sentiment_models[kind] = fit_sentiment(SentimentNet(len(review_data["stems"]), kind), review_data)

# bag-of-words baseline on the same stems
tfidf = TfidfVectorizer(analyzer=lambda tokens: tokens, min_df=2, max_features=MAX_VOCAB)
docs_train = [review_data["docs"][i] for i in review_data["train"]]
docs_val = [review_data["docs"][i] for i in review_data["val"]]
docs_test = [review_data["docs"][i] for i in review_data["test"]]
B_train, B_val, B_test = tfidf.fit_transform(docs_train), tfidf.transform(docs_val), tfidf.transform(docs_test)
y_train_np, y_val_np, y_test_np = (sentiment[review_data[s]] for s in ["train", "val", "test"])

best_c = max([0.3, 1, 3, 10, 30], key=lambda c: LogisticRegression(C=c, max_iter=2000).fit(B_train, y_train_np).score(B_val, y_val_np))
tfidf_logreg = LogisticRegression(C=best_c, max_iter=2000).fit(B_train, y_train_np)

rows = [("majority class", max(y_test_np.mean(), 1 - y_test_np.mean()), 0.5, "-")]
rows.append(("TF-IDF + logistic regression", tfidf_logreg.score(B_test, y_test_np),
             roc_auc_score(y_test_np, tfidf_logreg.predict_proba(B_test)[:, 1]), "-"))
for kind, (model, history) in sentiment_models.items():
    probs, truth = test_scores(model, review_data)
    rows.append((f"{kind.upper()} (hidden size 64)", ((probs > 0.5) == truth).mean(), roc_auc_score(truth, probs), history["best_epoch"] + 1))

print(f"{'model':<30} {'test accuracy':>14} {'test AUC':>9} {'best epoch':>11}")
for name, acc, auc, epoch in rows:
    print(f"{name:<30} {acc:>14.3f} {auc:>9.3f} {str(epoch):>11}")
print("selected C of the logistic regression:", best_c)

### Reading a Review Word by Word

In [ ]:
def word_by_word(model, text):
    words = preprocess(text)
    ids_ = torch.tensor([[review_data["stem_to_id"].get(w, 1) for w in words]])
    model.eval()
    with torch.no_grad():
        return words, torch.sigmoid(model.logits_after_each_word(ids_)).numpy()

gru = sentiment_models["gru"][0]
examples = ["The screen is great and the camera is nice but the battery is terrible and it does not hold a charge",
            "Not bad at all, works great and I would buy it again"]

fig, axes = plt.subplots(len(examples), 1, figsize=(10, 6))
for ax, text in zip(axes, examples):
    words, probs = word_by_word(gru, text)
    ax.plot(probs, marker="o")
    ax.axhline(0.5, color="gray", linestyle=":")
    ax.set(ylim=(0, 1), ylabel="P(positive)", xticks=range(len(words)))
    ax.set_xticklabels(words, rotation=60, fontsize=8)
    ax.set_title(text, fontsize=9)
plt.tight_layout()
plt.show()

### Does Keeping the Negations Matter?

### Where Does the Network Fail?

## Limitations

## Summary

## Exercises